# 🔹 Notebook 7 — Linguaggi di Query: SPARQL, Cypher, NL2Graph

**Obiettivo**: Eseguire la stessa query su SPARQL (Wikidata), Cypher (Neo4j), e generare query via NL2Graph con Ollama.

Corrisponde al capitolo **Interrogare un Knowledge Graph**.


## Setup

In [ ]:
import os, json
import httpx
from neo4j import GraphDatabase

NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_AUTH = ('neo4j', os.getenv('NEO4J_PASSWORD', 'yourpassword'))
OLLAMA_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')
LLM_MODEL = os.getenv('OLLAMA_LLM_MODEL', 'qwen2.5:14b')

driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
def cypher(q, **p):
    with driver.session() as s: return [r.data() for r in s.run(q, p)]
print('Neo4j:', cypher('RETURN 1 AS ok'))

## 7.1 — SPARQL su Wikidata: query su KG pubblico

In [ ]:
# Query Wikidata: citta' italiane con piu' di 500.000 abitanti
SPARQL_ENDPOINT = 'https://query.wikidata.org/sparql'

query = """
SELECT ?cityLabel ?population WHERE {
  ?city wdt:P31 wd:Q515 .         # istanza di citta'
  ?city wdt:P17 wd:Q38 .          # paese = Italia
  ?city wdt:P1082 ?population .   # popolazione
  FILTER (?population > 500000)
  SERVICE wikibase:label { bd:serviceParam wikibase:language 'it,en' . }
} ORDER BY DESC(?population)
LIMIT 10
"""

try:
    resp = httpx.get(SPARQL_ENDPOINT, params={'query': query, 'format': 'json'}, timeout=30)
    data = resp.json()['results']['bindings']
    print('Citta italiane > 500k abitanti (Wikidata SPARQL):')
    for row in data:
        print(f"  {row['cityLabel']['value']}: {int(row['population']['value']):,}")
except Exception as e:
    print(f'Errore (serve connessione internet): {e}')

## 7.2 — Cypher su Neo4j: stesse query del libro

In [ ]:
# Assicurati che il seed del Notebook 1 sia stato eseguito

# Query 1: Pattern matching - amici di Alice che lavorano in Tech
results = cypher("""
    MATCH (alice:Person {name:'Alice'})-[:KNOWS]->(friend:Person)
          -[:WORKS_AT]->(company:Company)
    RETURN friend.name AS amico, company.name AS azienda, company.industry AS settore
""")
print('Amici di Alice e dove lavorano:')
for r in results:
    print(f"  {r['amico']} @ {r['azienda']} ({r['settore']})")

# Query 2: Aggregazione - dipendenti per azienda
results = cypher("""
    MATCH (p:Person)-[:WORKS_AT]->(c:Company)
    RETURN c.name AS azienda, COUNT(p) AS dipendenti, COLLECT(p.name) AS nomi
    ORDER BY dipendenti DESC
""")
print('\nDipendenti per azienda:')
for r in results:
    print(f"  {r['azienda']}: {r['dipendenti']} ({', '.join(r['nomi'])})")

# Query 3: Shortest path
results = cypher("""
    MATCH path = shortestPath(
        (a:Person {name:'Alice'})-[:KNOWS*]-(d:Person {name:'Dave'}))
    RETURN length(path) AS gradi, [n IN nodes(path) | n.name] AS percorso
""")
print('\nShortest path Alice -> Dave:')
for r in results:
    print(f"  {r['gradi']} gradi: {' -> '.join(r['percorso'])}")

## 7.3 — NL2Graph: da linguaggio naturale a Cypher con Ollama

In [ ]:
def nl_to_cypher(question, schema):
    prompt = f"""Dato questo schema Neo4j:
{schema}

Converti la domanda in una query Cypher valida. Rispondi SOLO con la query, niente spiegazioni.

Domanda: {question}
Query Cypher:"""
    resp = httpx.post(f'{OLLAMA_URL}/api/chat', json={
        'model': LLM_MODEL, 'stream': False,
        'messages': [{'role':'user','content':prompt}]
    }, timeout=60)
    return resp.json()['message']['content'].strip()

# Schema del nostro KG
schema = """Nodi: Person(name, age, email), Company(name, industry, location), Technology(name)
Relazioni: WORKS_AT(since, role), KNOWS(since), SKILLED_IN(level), USES"""

# Test con 3 domande diverse
domande = [
    'Chi lavora a TechCorp?',
    'Quali tecnologie usa TechCorp?',
    'Chi conosce qualcuno che lavora in FinServ?',
]

for q in domande:
    print(f'Domanda: "{q}"')
    cypher_query = nl_to_cypher(q, schema)
    print(f'Cypher:  {cypher_query}')
    try:
        results = cypher(cypher_query)
        print(f'Risultato: {results[:3]}')
    except Exception as e:
        print(f'Errore esecuzione: {e}')
    print()

## 7.4 — NL2Graph con few-shot examples (accuracy boost)

In [ ]:
def nl_to_cypher_fewshot(question, schema, examples):
    examples_text = '\n'.join([f'Q: {e["q"]}\nA: {e["a"]}' for e in examples])
    prompt = f"""Schema Neo4j:
{schema}

Esempi di query corrette:
{examples_text}

Ora converti questa domanda in Cypher. Rispondi SOLO con la query.

Q: {question}
A:"""
    resp = httpx.post(f'{OLLAMA_URL}/api/chat', json={
        'model': LLM_MODEL, 'stream': False,
        'messages': [{'role':'user','content':prompt}]
    }, timeout=60)
    return resp.json()['message']['content'].strip()

# Few-shot examples
examples = [
    {'q': 'Chi lavora a TechCorp?', 'a': "MATCH (p:Person)-[:WORKS_AT]->(c:Company {name:'TechCorp'}) RETURN p.name"},
    {'q': 'Quanti amici ha Alice?', 'a': "MATCH (a:Person {name:'Alice'})-[:KNOWS]->(f) RETURN COUNT(f) AS amici"},
    {'q': 'Quali competenze ha Bob?', 'a': "MATCH (b:Person {name:'Bob'})-[:SKILLED_IN]->(t:Technology) RETURN t.name, b.name"},
]

# Test con domanda nuova
q = 'Trova esperti di Python che conoscono qualcuno in NewTech'
print(f'Domanda: "{q}"')
result = nl_to_cypher_fewshot(q, schema, examples)
print(f'Cypher (few-shot): {result}')
try:
    data = cypher(result)
    print(f'Risultato: {data}')
except Exception as e:
    print(f'Errore: {e}')

## Cleanup

In [ ]:
driver.close()
print('Notebook completato.')